# 04_extract_text

Dry-run text extraction on the latest inventory output. This notebook stays conservative: text-like files, PDF text extraction, DOCX paragraphs/tables, and XLSX sheet previews. No OCR yet.


In [1]:
from pathlib import Path
from datetime import datetime
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.inventory import ensure_inventory_schema
from src.extractors import ExtractConfig, enrich_inventory_with_text, save_text_outputs

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
inventory_files = [p for p in OUTPUT_DIR.glob('inventory_*.parquet') if not p.name.startswith('inventory_with_text_')]
assert inventory_files, 'No inventory parquet files found. Run 02_inventory.ipynb first.'
# Pick the newest file by filesystem timestamp, not filename order.
INVENTORY_PATH = max(inventory_files, key=lambda p: p.stat().st_mtime)
print('Using inventory file:', INVENTORY_PATH.name)


Using inventory file: inventory_Random_Files_WORKING_COPY_20260308_115612.parquet


In [2]:
inv = pd.read_parquet(INVENTORY_PATH)
inv = ensure_inventory_schema(inv)
print('Rows:', len(inv))
preview_cols = [c for c in ['relative_path', 'suffix', 'size_bytes'] if c in inv.columns]
display(inv[preview_cols].head(10))
if 'suffix' in inv.columns:
    display(inv['suffix'].fillna('').value_counts().rename_axis('suffix').reset_index(name='count').head(20))
else:
    print('suffix column unavailable after schema backfill')


Rows: 1806


,relative_path,suffix,size_bytes
0,01_selected\New Text Document.ogb,.ogb,0
1,01_selected\doclaynet_pdf\doclaynet_pdf_0001_r...,.pdf,16575
2,01_selected\doclaynet_pdf\doclaynet_pdf_0001_r...,.txt,3165
3,01_selected\doclaynet_pdf\doclaynet_pdf_0002_p...,.pdf,27076
4,01_selected\doclaynet_pdf\doclaynet_pdf_0002_p...,.txt,5
5,01_selected\doclaynet_pdf\doclaynet_pdf_0003_1...,.pdf,99867
6,01_selected\doclaynet_pdf\doclaynet_pdf_0003_1...,.txt,2414
7,01_selected\doclaynet_pdf\doclaynet_pdf_0004_N...,.pdf,273096
8,01_selected\doclaynet_pdf\doclaynet_pdf_0004_N...,.txt,5646
9,01_selected\doclaynet_pdf\doclaynet_pdf_0005_N...,.pdf,60106


,suffix,count
0,.pdf,682
1,.txt,353
2,.xls,213
3,.doc,165
4,.stp,156
5,.html,86
6,.png,75
7,.ppt,50
8,.csv,17
9,.rtf,5


In [3]:
config = ExtractConfig(
    max_chars_per_file=12000,
    max_csv_rows=30,
    max_csv_columns=20,
    max_xlsx_rows_per_sheet=30,
    max_xlsx_columns=20,
    max_docx_paragraphs=300,
    max_pdf_pages=30,
    preview_chars=300,
)
config


ExtractConfig(max_chars_per_file=12000, max_csv_rows=30, max_csv_columns=20, max_xlsx_rows_per_sheet=30, max_xlsx_columns=20, max_docx_paragraphs=300, max_pdf_pages=30, preview_chars=300)

In [4]:
enriched = enrich_inventory_with_text(inv, path_column='absolute_path', config=config)
display(enriched[['relative_path', 'suffix', 'text_status', 'text_source', 'extracted_chars', 'text_preview']].head(20))


invalid pdf header: b'\x00\x0bvol'
incorrect startxref pointer(3)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams
could not convert string to float: b'0.10699.' : FloatObject (b'0.10699.') invalid; use 0.0 instead
invalid pdf header: b'\x00\nvol'
incorrect startxref pointer(3)
parsing for Object Streams
invalid pdf header: b'\x00\rTM-'
incorrect startxref pointer(3)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams


,relative_path,suffix,text_status,text_source,extracted_chars,text_preview
0,01_selected\New Text Document.ogb,.ogb,unsupported,unsupported,0,
1,01_selected\doclaynet_pdf\doclaynet_pdf_0001_r...,.pdf,ok,pdf,3069,InnoDB Transaction Model • REPEATABLE READ Thi...
2,01_selected\doclaynet_pdf\doclaynet_pdf_0001_r...,.txt,ok,text,3129,InnoDB Transaction Model • REPEATABLE READ •Fo...
3,01_selected\doclaynet_pdf\doclaynet_pdf_0002_p...,.pdf,ok,pdf,5,10-12
4,01_selected\doclaynet_pdf\doclaynet_pdf_0002_p...,.txt,ok,text,5,10-12
5,01_selected\doclaynet_pdf\doclaynet_pdf_0003_1...,.pdf,ok,pdf,2161,"In the incoherent regime, the adiabatic elimin..."
6,01_selected\doclaynet_pdf\doclaynet_pdf_0003_1...,.txt,ok,text,2311,"In the incoherent regime, the adiabatic elimin..."
7,01_selected\doclaynet_pdf\doclaynet_pdf_0004_N...,.pdf,ok,pdf,5569,152 Non-Corporate CDOs and Other Derivative Tr...
8,01_selected\doclaynet_pdf\doclaynet_pdf_0004_N...,.txt,ok,text,5632,Non-Corporate CDOs and Other Derivative Transa...
9,01_selected\doclaynet_pdf\doclaynet_pdf_0005_N...,.pdf,ok,pdf,2235,-42- UTILIZATION: Fiscal Year 2010 First Secon...


In [7]:
display(enriched[enriched['text_status'] == 'error'][['relative_path', 'suffix', 'text_error']])

,relative_path,suffix,text_error
678,01_selected\govdocs1\govdocs1_0068_002400.pdf,.pdf,PdfReadError: Invalid Elementary Object starti...


In [8]:
display(enriched['text_status'].value_counts(dropna=False).rename_axis('text_status').reset_index(name='count'))
display(enriched['text_source'].value_counts(dropna=False).rename_axis('text_source').reset_index(name='count'))
display(enriched[enriched['text_status'] == 'error'][['relative_path', 'suffix', 'text_error']].head(20))


,text_status,count
0,ok,825
1,unsupported,666
2,empty,314
3,error,1


,text_source,count
0,pdf,682
1,unsupported,666
2,text,458


,relative_path,suffix,text_error
678,01_selected\govdocs1\govdocs1_0068_002400.pdf,.pdf,PdfReadError: Invalid Elementary Object starti...


In [9]:
display(enriched[enriched['has_extracted_text']][['relative_path', 'text_source', 'extracted_chars', 'text_preview']].head(20))


,relative_path,text_source,extracted_chars,text_preview
1,01_selected\doclaynet_pdf\doclaynet_pdf_0001_r...,pdf,3069,InnoDB Transaction Model • REPEATABLE READ Thi...
2,01_selected\doclaynet_pdf\doclaynet_pdf_0001_r...,text,3129,InnoDB Transaction Model • REPEATABLE READ •Fo...
3,01_selected\doclaynet_pdf\doclaynet_pdf_0002_p...,pdf,5,10-12
4,01_selected\doclaynet_pdf\doclaynet_pdf_0002_p...,text,5,10-12
5,01_selected\doclaynet_pdf\doclaynet_pdf_0003_1...,pdf,2161,"In the incoherent regime, the adiabatic elimin..."
6,01_selected\doclaynet_pdf\doclaynet_pdf_0003_1...,text,2311,"In the incoherent regime, the adiabatic elimin..."
7,01_selected\doclaynet_pdf\doclaynet_pdf_0004_N...,pdf,5569,152 Non-Corporate CDOs and Other Derivative Tr...
8,01_selected\doclaynet_pdf\doclaynet_pdf_0004_N...,text,5632,Non-Corporate CDOs and Other Derivative Transa...
9,01_selected\doclaynet_pdf\doclaynet_pdf_0005_N...,pdf,2235,-42- UTILIZATION: Fiscal Year 2010 First Secon...
10,01_selected\doclaynet_pdf\doclaynet_pdf_0005_N...,text,2231,UTILIZATION: Fiscal Year 2010 First Second Thi...


In [10]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_base = OUTPUT_DIR / f'inventory_with_text_{timestamp}'
csv_path, parquet_path = save_text_outputs(enriched, output_base)
print('Saved CSV   :', csv_path)
print('Saved Parquet:', parquet_path)


Saved CSV   : c:\00_Developement\sch-file-organizer\data\outputs\inventory_with_text_20260308_144155.csv
Saved Parquet: c:\00_Developement\sch-file-organizer\data\outputs\inventory_with_text_20260308_144155.parquet


In [5]:
err = enriched[enriched['text_status'] == 'error'].copy()
long_err = err[err['path_length'] > 250].copy()
fnf_long = long_err[long_err['text_error'].fillna('').str.contains('FileNotFoundError', regex=False)]
print('errors_total =', len(err))
print('errors_path_gt_250 =', len(long_err))
print('FileNotFoundError_on_path_gt_250 =', len(fnf_long))
display(long_err[['relative_path', 'path_length', 'suffix', 'text_error']].head(20))

errors_total = 1
errors_path_gt_250 = 0
FileNotFoundError_on_path_gt_250 = 0


,relative_path,path_length,suffix,text_error
